In [18]:
from pathlib import Path
import numpy as np
import pandas as pd
import cv2


full_file_path = Path(r"Z:\Jasmine_Laurence\Experimental_Data\JR_test2026")
TTL_bin_path = r"Z:\Jasmine_Laurence\Experimental_Data\JR_test2026\test\ses-001_cordoptimiz-111_g0\ses-001_cordoptimiz-111_g0_imec0\ses-001_cordoptimiz-111_g0_t0.exported.imec0.ap.bin"


AI_file = list(full_file_path.glob("*analog.bin"))[0]
daq_sampling_rate = 15000
if '.bin' in str(AI_file): 
    AI_data = np.fromfile(AI_file)

In [ ]:
# audio stims?

audio_data = AI_data[np.arange(1, len(AI_data), 4)] # four interleaved time series
audio_num_samples = len(audio_data)
audio_on = abs(audio_data)>3
data_on_idx = np.where(audio_on)[0]
idx_since_data_on = np.append(np.inf, np.diff(data_on_idx))
data_onset_idx = data_on_idx[idx_since_data_on > int(daq_sampling_rate*5)]
print("the number of audio stims is: ", len(data_onset_idx))

IndexError: boolean index did not match indexed array along axis 0; size of axis is 0 but size of corresponding boolean axis is 1

In [16]:
# frames check
frames_csv_path = list(full_file_path.glob("*frames.csv"))[0]

frames_csv = pd.read_csv(frames_csv_path, names=['frame number', 'zero', 'timestamp'])
print(f"Number of expected frames: {len(frames_csv['timestamp'])}")

camera_trigger_data = AI_data[np.arange(0, len(AI_data), 4)] # four interleaved time series
frame_trigger_onsets = np.diff(camera_trigger_data)
frame_trigger_onsets_idx = np.where(frame_trigger_onsets > 1)[0] + 1
print("Number of camera trigger samples: ", len(frame_trigger_onsets_idx))

video_file = list(full_file_path.glob("*_cam.avi"))[0]
video_object = cv2.VideoCapture(video_file)
num_frames = int(video_object.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Number of recorded camera frames : {num_frames}")

Number of expected frames: 2090
Number of camera trigger samples:  2090
Number of recorded camera frames : 2090


In [22]:
# spikeglx TTL check
def unpackbits(exported_sync_channel_imec, num_bits=16, bit_filter=6):
    """This is a yulin wizard function that extracts the imec digital signal from the
    exported imec file. You must use spikeglx viewer to export only the 384 sync channel.
    It unpacks the bin file in bits and returns a digital signal of 0s and 1s.
    """
    xshape = list(exported_sync_channel_imec.shape)
    x = exported_sync_channel_imec.reshape([-1, 1])
    to_and = 2 ** np.arange(num_bits).reshape([1, num_bits])
    temp = (x & to_and).reshape(xshape + [num_bits])
    if bit_filter is not None:
        temp = temp[:, 6]
    temp[temp > 0] = 1
    digital_signal = temp
    return digital_signal

def get_onset_offset(signal, threshold, clean=True):
    """
    Get onset/offset times when a signal (either bonsai or imec TLL depending on argument)
    goes below>above and above>below a given threshold. If no starts or ends kill programme.

    Arguments:
        signal: 1d numpy array - bonsai or imec TLL
        thhreshold: float, threshold
        clean: bool. If true ends before the first start and starts after the last end are removed
        type: imec or bonsai

    Returns:
        Starts: Indexes of pulse onsets
        Ends: Indexes of pulse offsets
    """

    above = np.zeros_like(signal)
    above[signal >= threshold] = 1  # If the signal is above voltage threshold, set to 1
    if np.sum(np.isnan(signal)) > 0:
        above[np.isnan(signal)] = np.nan
    der = np.diff(above, n=1, axis=0)  # Create an array of differences
    starts = np.where(der > 0.5)[0]  # Where does the signal switch from 0 to 1
    ends = np.where(der < -0.5)[0]  # Where does the signal switch from 1 to 0

    if clean:

        # offsets before the first onsets are removed
        ends = np.array([e for e in ends if e > starts[0]])

        # onsets before the last offsets are removed
        if np.any(ends):
            starts = np.array([s for s in starts if s < ends[-1]])

    if not np.any(starts):
        assert False, "No onsets"
    if not np.any(ends):
        assert False, "No offsets"

    return starts, ends

def check_for_abberant_pulses(bonsai_sync_onsets, ephys_sync_onsets, sampling_rate, delete = True):
    """A function that checks if the delta between onsets and offsets is not greater or less than
    what should roughly be expected. Are pulse lengths to be expected?

    Args:
        bonsai_sync_onsets (_type_): _description_
        ephys_sync_onsets (_type_): _description_
        sampling_rate (_type_): _description_
        delete (bool): If True, remove the pulse onsets that are too brief, this is likely a result of a bad sync signal. This will hopefully fix alignment issues.
            if False, just log the error. 
            
            If syncing not working, FIRST SET TO FALSE, then check the onsets and offsets to see if they are correct.
    """

    # log

    # Bonsai pulse length check if too long or short
    bonsai_pulse_len_under_errors = np.where(np.diff(bonsai_sync_onsets) < (sampling_rate / 2))[0]  # Is onset delta less than 15khz
    bonsai_pulse_len_over_errors = np.where(np.diff(bonsai_sync_onsets) > (sampling_rate * 1.5))[0]  # Is onset delta greater than 15khz

    if bonsai_pulse_len_under_errors.any():
        onsets_delta = np.diff(bonsai_sync_onsets)
        counts = {k: len(onsets_delta[onsets_delta == k]) for k in set(onsets_delta)}
        print(f"There are {len(bonsai_pulse_len_under_errors)} bonsai pulses that are less than 1hz duration")
        print(f"Bonsai pulse less than 1hz duration: {counts}")
        
        if delete:
            print("Removing bonsai pulses onsets that are too brief, this is likely a result of a bad sync signal. This will hopefully fix alignment issues.")
            bonsai_sync_onsets = np.delete(bonsai_sync_onsets, bonsai_pulse_len_under_errors)

    if bonsai_pulse_len_over_errors.any():
        print("Bonsai pulse greater than 1hz duration")

    # Imec pulse length check if too long or short
    imec_pulse_len_under_errors = np.where(np.diff(ephys_sync_onsets) < (sampling_rate / 2))[0]  # Is onset delta less than 15khz
    imec_pulse_len_over_errors = np.where(np.diff(ephys_sync_onsets) > (sampling_rate * 1.5))[0]  # Is onset delta greater than 15khz

    if imec_pulse_len_over_errors.any():
        print("Bonsai pulse greater than 1hz duration")
        
    if imec_pulse_len_under_errors.any():
        onsets_delta = np.diff(ephys_sync_onsets)
        counts = {k: len(onsets_delta[onsets_delta == k]) for k in set(onsets_delta)}
        print(f"There are {len(imec_pulse_len_under_errors)} imec pulses that are less than 1hz duration")
        print(f"Imec pulses less than 1hz duration: {counts}")
        
        if delete:
            print("Removing imec pulses onsets that are too brief, this is likely a result of a bad sync signal. This will hopefully fix alignment issues.")
            ephys_sync_onsets = np.delete(ephys_sync_onsets, imec_pulse_len_under_errors)
    
    return ephys_sync_onsets, bonsai_sync_onsets

sampling_rate = 30000
bonsai_ttl = AI_data[np.arange(3, len(AI_data), 4)]
imec_TTL = unpackbits(np.fromfile(Path(TTL_bin_path), dtype=np.int16), bit_filter=6)

bonsai_sync_onsets, bonsai_sync_offsets = get_onset_offset(bonsai_ttl, 2.5)
ephys_sync_onsets, ephys_sync_offsets = get_onset_offset(imec_TTL, 0.5)

# Check pulse lengths
ephys_sync_onsets, bonsai_sync_onsets = check_for_abberant_pulses(bonsai_sync_onsets, ephys_sync_onsets, sampling_rate)

print(f"The number of efizz pulses {len(ephys_sync_onsets)} onsets should match the number of bonsai pulses {len(bonsai_sync_onsets)} onsets.")

The number of efizz pulses 52 onsets should match the number of bonsai pulses 52 onsets.
